# Marker Repo - calculations: Scores, homology

In [ ]:
%load_ext autoreload
%autoreload 2

In this notebook, individual markers can be weighted using various functions. In addition, BioMart or the HomoloGene DB enable genes to be transferred from a source organism to a target organism. Thus, even analyses with specific organisms can be performed without already existing suitable lists.

## Loading packages

In [ ]:
import markerrepo.marker_repo as mr
import markerrepo.calculations as calc
import markerrepo.utils as utils
import markerrepo.wrappers as wrap
import markerrepo.plotting as plot

# Calculate scores

This part compares and scores markers from selected marker lists using <b>ubiquitousness index</b>.
A score of '0' signifies that the marker is the most specific within this selection, 
while a score of '1' indicates that the marker is the most prevalent.

Start by selecting lists you want to compare.

In [ ]:
results = mr.guided_search(out="marker_list")

Compare the lists and show the new DataFrame.

In [ ]:
results_scored = calc.compare_marker_lists(marker_df=results)
display(results_scored)

With this function all marker scores, which are part of the <b>panglaoDB</b>, are being updated by using the calculated ubiquitousness index from the panglaoDB. The resulting DataFrame thus contains the dynamically calculated scores as well as scores of the panglaoDB - depending on whether the corresponding gene is present in the panglaoDB or not.

In [ ]:
results_panglao = calc.update_scores(df=results_scored)
display(results_panglao)

Finally the scored list can be exported.

In [ ]:
mr.export_marker_list(results_panglao)

# Transfer markers using homology

In this section, with the use of homology, marker genes can be transferred from a source organism to a target organism. Two different approaches are currently available: The use of <b>BioMart and Ensemble</b> and the use of <b>HomoloGene db</b>. In the first part of this section, the currently supported organisms of each approach are presented. Based on this, a decision can be made for one or the other approach. Of course, both approaches can also be performed, but then it is no longer guaranteed that the selected organisms are actually supported.

Pull whitelist repository, update if necessary and get supported organisms.

In [ ]:
mr.get_whitelists()
biomart_orgs = calc.get_supported_biomart_organisms()
homologene_orgs = calc.get_supported_taxonomy_ids()

Check which organisms are currently supported by BioMart or HomoloGene and select one of them.

In [ ]:
db_choice = calc.select_db(biomart_orgs, homologene_orgs)

if db_choice == "biomart":
    organisms = biomart_orgs
else:
    organisms = homologene_orgs

Select source organism and target organism.

In [ ]:
source_organism, source_tax = mr.select(whitelist=organisms, heading="source organism").split(" ")
target_organism, target_tax = mr.select(whitelist=organisms, heading="target organism").split(" ")  
source_genes = utils.read_whitelist(f"genes/{source_organism}")['whitelist']
target_genes = utils.read_whitelist(f"genes/{target_organism}")['whitelist']

## Transfer markers from one organism to another using BioMart

Select available source and target organism identifier from biomart db.

In [ ]:
source_organism_bm = mr.select(whitelist=calc.get_dataset_names(source_organism), heading="BioMart source organism")
target_organism_bm = mr.select(whitelist=calc.get_dataset_names(target_organism), heading="BioMart target organism")

Fetch necessary data from BioMart.

In [ ]:
biomart_db = calc.fetch_homologs(source_organism_bm, target_organism_bm).dropna()

Select all lists of source organism for cell type annotation.

In [ ]:
keywords = {"Organism name": source_organism, "List type": "Cell type annotation"}
source_df = mr.search_db(mr.get_db(), keywords, case_sensitive=True, exact=True, out="marker_list")

Or select lists using guided search.

In [ ]:
source_df = mr.guided_search(out="marker_list")

In [ ]:
display(source_df)

Transfer source markers to target markers using BioMart results and extend transferred markers.

In [ ]:
transferred_list = calc.transfer_markers_biomart(biomart_db, source_df, target_genes,
                                                 source_whitelist=source_genes, calc_proportions=True, plots=True)
print("Transferred markers:")
display(transferred_list)

Export marker list

In [ ]:
mr.export_marker_list(markers_extended, file_name="human-mouse_hcm_cta", marker_id="symbol")

## Transfer markers from one organism to another using HomoloGene db

In [ ]:
print(f"Source organism: {source_organism}\nTarget organism: {target_organism}")

Get HomoloGene db.

In [ ]:
hg_db = calc.download_homologene_data()

Select all lists of source organism for cell type annotation.

In [ ]:
keywords = {"Organism name": source_organism, "List type": "Cell type annotation"}
source_df = mr.search_db(mr.get_db(), keywords, case_sensitive=True, exact=True, out="marker_list")

Or select lists using guided search.

In [ ]:
source_df = mr.guided_search(out="marker_list")

In [ ]:
display(source_df)

Transfer source markers to target markers using HomoloGene db and extend transferred markers.

In [ ]:
transferred_list = calc.transfer_markers(source_df, source_tax, target_tax, hg_db, target_genes,
                                         source_whitelist=source_genes, calc_proportions=True, plots=True)
print("Transferred markers:")
display(transferred_list)

Export marker list

In [ ]:
mr.export_marker_list(markers_extended, file_name="human-mouse_panglao_cta_homologene", marker_id="symbol")